# Random Walk and Autocorrelation

Every annualization in this repository, $\sigma\sqrt{T}$ and $\mu T$, assumed that returns are independent over time: prices follow a random
walk. This notebook makes that assumption testable. It defines stationarity and autocorrelation, derives the standard tests
(autocorrelation bands, Ljung–Box, variance ratio), and looks at the two places where autocorrelation matters most in
practice: smoothed returns of illiquid assets, and the strong autocorrelation of squared returns that leads to volatility models (09/02).

## 0. Math link

There is no time-series notebook in the course repos yet (curriculum gap). This notebook builds on:

- [Covariance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/06_covariance.ipynb)
  and [correlation](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/09_correlation_coefficient.ipynb):
  autocorrelation is the correlation of a series with its own past.
- [Variance of sums](../03_covariance_and_two_assets/01_variance_of_sum_and_difference.ipynb): the variance of a multi-period return
  includes all the cross-period covariances.
- [Chi-squared distribution](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/11_chi_squared_distribution.ipynb):
  the Ljung–Box statistic is a sum of squared, approximately normal autocorrelations.
- [Central limit theorem](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/04_sampling_and_point_estimation/07_central_limit_theorem_continuous.ipynb).

## 1. Introduction

If returns were predictable from their own past, a simple rule would beat the market and the $\sqrt T$ rule for risk would be wrong.
Liquid asset prices come close to a random walk in their returns, yet two kinds of dependence are important: returns of
appraisal-based assets (real estate, private equity) are smoothed and strongly autocorrelated, and the **size** of returns
is persistent even when their sign is not.

After this notebook you can:

- define a random walk and weak stationarity, and tell which financial series are stationary;
- compute and test sample autocorrelations with confidence bands and the Ljung–Box statistic;
- derive the autocorrelation function of an AR(1) process;
- compute variance ratios and explain how autocorrelation breaks the $\sqrt T$ rule;
- unsmooth appraisal-based returns;
- recognize spurious regressions between price levels.

## 2. Theory

### Random walk

The log price $p_t$ follows a random walk with drift if

$$
p_t = p_{t-1} + \mu + \varepsilon_t, \qquad \varepsilon_t \text{ independent, mean 0, variance } \sigma^2
$$

The log return $r_t = p_t - p_{t-1} = \mu + \varepsilon_t$ is i.i.d. Over $h$ periods, $p_{t+h} - p_t$ has mean $h\mu$ and variance $h\sigma^2$: the $\sqrt h$ rule. A weaker
version only requires $E[\varepsilon_t \mid \text{past}] = 0$ (a martingale difference): returns unpredictable in mean, but their variance may
depend on the past.

### Stationarity

A series $x_t$ is **weakly stationary** if $E[x_t] = \mu$, $\operatorname{Var}(x_t) = \gamma_0$ and $\operatorname{Cov}(x_t, x_{t-k}) = \gamma_k$ do not depend on $t$. Returns are (approximately)
stationary; price levels are not, since their variance grows with $t$.

### Autocorrelation

The autocorrelation at lag $k$ is $\rho_k = \gamma_k/\gamma_0$. The sample version with $n$ observations is

$$
\hat\rho_k = \frac{\sum_{t=k+1}^n (x_t - \bar x)(x_{t-k} - \bar x)}{\sum_{t=1}^n (x_t - \bar x)^2}
$$

For an i.i.d. series, $\hat\rho_k$ is approximately $N(0, 1/n)$, so $\pm 1.96/\sqrt n$ bands mark individually significant autocorrelations. With
many lags, about 5% fall outside the bands by chance.

### Ljung–Box test

To test the first $m$ autocorrelations jointly:

$$
Q = n(n+2)\sum_{k=1}^m \frac{\hat\rho_k^2}{n - k} \;\approx\; \chi^2_m \text{ under i.i.d.}
$$

since each $\sqrt n\,\hat\rho_k$ is approximately standard normal and they are approximately independent.

### AR(1)

$x_t = c + \phi x_{t-1} + \varepsilon_t$ with $|\phi| < 1$ is stationary with

$$
E[x] = \frac{c}{1-\phi}, \qquad \gamma_0 = \frac{\sigma^2}{1-\phi^2}, \qquad \rho_k = \phi^k
$$

(Multiply $x_t - E[x] = \phi(x_{t-1} - E[x]) + \varepsilon_t$ by $x_{t-k} - E[x]$ and take expectations: $\gamma_k = \phi\gamma_{k-1}$.)

### Variance ratio

The variance of a $q$-period return is $\operatorname{Var}(\sum_{j=0}^{q-1} r_{t-j}) = \gamma_0\big[q + 2\sum_{k=1}^{q-1}(q - k)\rho_k\big]$. The **variance ratio**

$$
\mathrm{VR}(q) = \frac{\operatorname{Var}(r_t^{(q)})}{q\operatorname{Var}(r_t)} = 1 + 2\sum_{k=1}^{q-1}\Big(1 - \frac kq\Big)\rho_k
$$

equals 1 for a random walk, exceeds 1 with positive autocorrelation (trending, momentum), and is below 1 with negative autocorrelation
(mean reversion). Lo and MacKinlay (1988): under i.i.d. returns, $\widehat{\mathrm{VR}}(q) - 1$ has standard error $\sqrt{2(2q-1)(q-1)/(3qn)}$. The $\sqrt T$ scaling of
volatility is off by the factor $\sqrt{\mathrm{VR}(q)}$.

### Smoothed returns

Appraisal-based returns adjust slowly: $r^{\text{obs}}_t = (1-\theta)r_t + \theta r^{\text{obs}}_{t-1}$ with true i.i.d. returns $r_t$. The observed series is AR(1) with $\phi = \theta$ and

$$
\operatorname{Var}(r^{\text{obs}}) = \sigma^2\frac{1-\theta}{1+\theta}
$$

so it understates true volatility. **Unsmoothing** (Geltner 1993) inverts the filter: $r_t = (r^{\text{obs}}_t - \theta r^{\text{obs}}_{t-1})/(1-\theta)$.

### Spurious regression

Regressing one random-walk **level** on another independent one produces large $R^2$ and "significant" slopes far more often than 5%
(Granger and Newbold 1974): both series wander, and the regression mistakes common wandering for a relationship. Regress returns,
not prices.

### Squared returns

Returns can be uncorrelated while $r_t^2$ or $|r_t|$ are strongly autocorrelated: calm and turbulent periods cluster. This is
the starting point of 09/02.

## 3. Worked Examples

### Example 1: the square-root rule

Daily volatility 1% under a random walk: after 252 days $1\%\sqrt{252} = 15.9\%$; after 21 days $4.6\%$.

### Example 2: autocorrelation bands

$n = 1{,}000$ daily returns: bands $\pm 1.96/\sqrt{1000} = \pm 0.062$. With 120 monthly returns: $\pm 0.179$.

### Example 3: variance ratio of an AR(1)

Monthly returns with $\phi = 0.2$ and $q = 12$:

$$
\mathrm{VR}(12) = 1 + 2\sum_{k=1}^{11}\Big(1 - \frac{k}{12}\Big)0.2^k = 1 + 2(0.1833 + 0.0333 + 0.0060 + 0.0011 + 0.0002 + \dots) = 1.448
$$

Annual volatility computed as monthly volatility × $\sqrt{12}$ is too low by the factor $\sqrt{1.448} = 1.20$.

### Example 4: Ljung–Box by hand

$n = 500$, first five sample autocorrelations $0.08, 0.02, -0.05, 0.03, 0.01$:

$$
Q = 500\cdot 502\Big(\frac{0.0064}{499} + \frac{0.0004}{498} + \frac{0.0025}{497} + \frac{0.0009}{496} + \frac{0.0001}{495}\Big) = 251{,}000\cdot 2.07\cdot 10^{-5} = 5.19
$$

The 95% quantile of $\chi^2_5$ is 11.07: no evidence against independence. Individually, $\hat\rho_1 = 0.08$ is also inside its band $\pm 1.96/\sqrt{500} = \pm 0.088$.

### Example 5: unsmoothing

Observed real estate returns: volatility 6%, lag-1 autocorrelation 0.6. True volatility $6\%\sqrt{(1 + 0.6)/(1 - 0.6)} = 6\%\cdot 2 = 12\%$.

## 4. Finance Application

### Where it matters

- **Risk scaling.** Daily VaR scaled by $\sqrt{10}$ to a 10-day horizon, or monthly volatility scaled by $\sqrt{12}$, is valid only without
  autocorrelation. Positive autocorrelation makes the scaled number too low.
- **Illiquid and appraisal-based assets.** Real estate, private equity, infrastructure and some hedge fund indices report smoothed
  returns. Their volatility and correlations with listed markets look low, and their Sharpe ratios high; unsmoothing
  corrects this before they enter an optimizer.
- **Trading signals.** Momentum and reversal strategies bet on non-zero autocorrelation; tests must account for the many lags and
  assets examined (06/03).
- **Spurious relationships.** Regressing price levels (or index levels on macro levels) produces impressive and meaningless fits.

### The usual misreadings

- "No autocorrelation, so returns are independent." Squared returns can still be strongly dependent.
- "The hedge fund index has a Sharpe ratio of 1.5 and 4% volatility." With lag-1 autocorrelation of 0.35, the unsmoothed volatility is about
  45% higher and the Sharpe ratio correspondingly lower.
- "Gold and the S&P 500 have an $R^2$ of 0.8 over twenty years." Probably levels; the relationship between their returns may be negligible.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
trading_days = 252

### Tools: autocorrelation, Ljung–Box, variance ratio

In [ ]:
def acf(x, max_lag):
    x = np.asarray(x) - np.mean(x)
    denom = x @ x
    return np.array([x[k:] @ x[:-k] / denom for k in range(1, max_lag + 1)])


def ljung_box(x, m):
    n = len(x)
    r = acf(x, m)
    Q = n * (n + 2) * np.sum(r**2 / (n - np.arange(1, m + 1)))
    return Q, stats.chi2.sf(Q, m)


def variance_ratio(x, q):
    # overlapping q-period sums; returns VR and Lo-MacKinlay z-statistic (homoskedastic)
    x = np.asarray(x) - np.mean(x)
    n = len(x)
    var1 = x @ x / (n - 1)
    sums = np.convolve(x, np.ones(q), mode="valid")
    varq = sums @ sums / (len(sums) - 1)
    vr = varq / (q * var1)
    se = np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * n))
    return vr, (vr - 1) / se

### Three daily series

Ten years of daily returns: (a) i.i.d. normal, (b) AR(1) with $\phi = 0.1$, (c) uncorrelated returns with clustered volatility
(a GARCH(1,1) process, 09/02 explains it). All have annual volatility about 16%.

In [ ]:
n = 10 * trading_days
sig_d = 0.16 / np.sqrt(trading_days)

r_iid = sig_d * rng.standard_normal(n)

phi = 0.1
r_ar = np.empty(n)
r_ar[0] = sig_d * rng.standard_normal()
for t in range(1, n):
    r_ar[t] = phi * r_ar[t - 1] + sig_d * np.sqrt(1 - phi**2) * rng.standard_normal()

omega, alpha_g, beta_g = sig_d**2 * (1 - 0.09 - 0.90), 0.09, 0.90
h = np.empty(n)
r_garch = np.empty(n)
h[0] = sig_d**2
for t in range(n):
    if t > 0:
        h[t] = omega + alpha_g * r_garch[t - 1] ** 2 + beta_g * h[t - 1]
    r_garch[t] = np.sqrt(h[t]) * rng.standard_normal()

band = 1.96 / np.sqrt(n)
rows = []
for name, x in [("i.i.d.", r_iid), ("AR(1) phi = 0.1", r_ar), ("GARCH", r_garch)]:
    a = acf(x, 5)
    a2 = acf(x**2, 5)
    Q, p = ljung_box(x, 10)
    Q2, p2 = ljung_box(x**2, 10)
    rows.append({"series": name, "ann. vol": x.std() * np.sqrt(trading_days), "rho_1": a[0], "rho_2": a[1],
                 "LB(10) p-value": p, "rho_1 of r^2": a2[0], "LB(10) p-value of r^2": p2})
print(f"95% band for single autocorrelations: +/-{band:.3f}")
pd.DataFrame(rows).set_index("series").round(4)

The AR(1) series shows $\hat\rho_1 = 0.10$, well outside the $\pm 0.039$ band, and Ljung–Box rejects clearly. The i.i.d. series happens to show
$\hat\rho_1 = 0.046$ and a Ljung–Box p-value of 0.03: a false positive of the kind a 5% test produces one time in twenty. The GARCH returns
show no autocorrelation (p = 0.61), but their **squares** have $\hat\rho_1 = 0.18$ and a p-value of zero: independent in sign, dependent in size.

### How often does Ljung–Box reject?

Size under i.i.d. normal and fat-tailed $t_3$ returns, and power against AR(1), for 1,000 series of 500 daily returns, $m = 10$, 5% level.

In [ ]:
def ar1_series(phi_, n_, rng_):
    e = rng_.standard_normal(n_)
    x = np.empty(n_)
    x[0] = e[0] / np.sqrt(1 - phi_**2)
    for t in range(1, n_):
        x[t] = phi_ * x[t - 1] + e[t]
    return x

rej = {}
rej["i.i.d. normal"] = np.mean([ljung_box(rng.standard_normal(500), 10)[1] < 0.05 for _ in range(1_000)])
rej["i.i.d. t(3)"] = np.mean([ljung_box(rng.standard_t(3, 500), 10)[1] < 0.05 for _ in range(1_000)])
for ph in [0.05, 0.1, 0.2]:
    rej[f"AR(1) phi = {ph}"] = np.mean([ljung_box(ar1_series(ph, 500, rng), 10)[1] < 0.05 for _ in range(1_000)])
pd.Series(rej).map(lambda v: f"{v:.1%}").rename("rejection rate")

The test holds its 5% level roughly (6.4% for normal, 5.3% for fat-tailed $t_3$ returns). Its power against small autocorrelation is low with 500
observations: 9% for $\phi = 0.05$, 25% for $\phi = 0.1$, 89% for $\phi = 0.2$.

### Variance ratios

In [ ]:
qs = [2, 5, 10, 21, 63]
vr_rows = []
for name, x in [("i.i.d.", r_iid), ("AR(1) phi = 0.1", r_ar), ("GARCH", r_garch)]:
    for q in qs:
        vr, z = variance_ratio(x, q)
        theory = 1 + 2 * sum((1 - k / q) * (phi**k if name.startswith("AR") else 0.0) for k in range(1, q))
        vr_rows.append({"series": name, "q": q, "VR": vr, "z": z, "theory": theory})
vr_table = pd.DataFrame(vr_rows).pivot(index="q", columns="series", values=["VR", "theory", "z"])
vr_table.round(3)

For the AR(1) series the sample variance ratios follow the theory (1.10 at two days, 1.21 at 21 days), with significant z-statistics at short
horizons. The i.i.d. series shows a variance ratio of 1.05 at $q = 2$ ($z = 2.3$), the same chance positive $\hat\rho_1$ as above. The GARCH series stays near 1:
volatility clustering does not change variance ratios, because the returns are uncorrelated.

### Smoothed real estate returns

True monthly returns of a property portfolio: i.i.d. with 12% annual volatility and 5% expected excess return. Reported returns are
smoothed with $\theta = 0.6$. Thirty years of data.

In [ ]:
n_m = 360
theta = 0.6
r_true = 0.05 / 12 + 0.12 / np.sqrt(12) * rng.standard_normal(n_m)
r_obs = np.empty(n_m)
r_obs[0] = r_true[0]
for t in range(1, n_m):
    r_obs[t] = (1 - theta) * r_true[t] + theta * r_obs[t - 1]

theta_hat = acf(r_obs, 1)[0]
r_unsm = (r_obs[1:] - theta_hat * r_obs[:-1]) / (1 - theta_hat)

def ann(x):
    return x.mean() * 12, x.std(ddof=1) * np.sqrt(12)

for name, x in [("true", r_true), ("observed (smoothed)", r_obs), ("unsmoothed", r_unsm)]:
    m_, s_ = ann(x)
    print(f"{name:20s}: mean {m_:6.2%}, vol {s_:6.2%}, Sharpe {m_ / s_:5.2f}, rho_1 {acf(x, 1)[0]:5.2f}")
print(f"estimated theta {theta_hat:.3f};  vol ratio theory sqrt((1-theta)/(1+theta)) = {np.sqrt((1 - theta) / (1 + theta)):.3f}")

Smoothing halves the measured volatility (6.4% against 12.4% true, theory factor 0.50) and doubles the Sharpe ratio (0.96 against 0.49),
while the mean is unaffected. Unsmoothing with the estimated $\theta = 0.64$ recovers a volatility of 13.6% and a Sharpe ratio of 0.44, and
removes the autocorrelation.

### Spurious regression

1,000 pairs of independent random walks of 120 monthly steps. Regress one on the other in levels and in returns.

In [ ]:
r2_lev, r2_ret, sig_lev = [], [], []
for _ in range(1_000):
    a_ = rng.standard_normal(120)
    b_ = rng.standard_normal(120)
    la, lb = np.cumsum(a_), np.cumsum(b_)
    res_l = stats.linregress(la, lb)
    r2_lev.append(res_l.rvalue**2)
    sig_lev.append(res_l.pvalue < 0.05)
    r2_ret.append(stats.linregress(a_, b_).rvalue ** 2)
print(f"levels : median R^2 {np.median(r2_lev):.2f}, 'significant' at 5% in {np.mean(sig_lev):.0%} of pairs")
print(f"returns: median R^2 {np.median(r2_ret):.3f}")

Between independent random-walk levels, the median $R^2$ is 0.18 and the slope is "significant" at the 5% level in 80% of pairs. Between their
returns the median $R^2$ is 0.004, and the test behaves as it should.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
t_ = np.arange(1, n + 1) / trading_days
for i in range(20):
    ax.plot(t_, np.cumsum(sig_d * rng.standard_normal(n)), color="grey", alpha=0.4, linewidth=0.7)
ax.plot(t_, 1.96 * 0.16 * np.sqrt(t_), color="tab:red", label="±1.96 σ√t")
ax.plot(t_, -1.96 * 0.16 * np.sqrt(t_), color="tab:red")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Random walks of the log price (16% annual vol, zero drift)")
ax.set_xlabel("years")
ax.set_ylabel("cumulative log return")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
lags = np.arange(1, 21)
for ax, (name, x) in zip(axes, [("i.i.d. returns", r_iid), ("AR(1) phi = 0.1 returns", r_ar), ("GARCH: squared returns", r_garch**2)]):
    ax.bar(lags, acf(x, 20), width=0.6)
    ax.axhline(band, color="tab:red", linestyle="--", linewidth=0.8)
    ax.axhline(-band, color="tab:red", linestyle="--", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("lag (days)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("sample autocorrelation")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
q_grid = np.arange(2, 37)
for ph, c in [(0.2, "tab:red"), (0.0, "black"), (-0.2, "tab:blue"), (0.6, "tab:orange")]:
    vr_th = [1 + 2 * sum((1 - k / q) * ph**k for k in range(1, q)) for q in q_grid]
    ax.plot(q_grid, vr_th, color=c, label=f"AR(1) phi = {ph}" + (" (smoothed real estate)" if ph == 0.6 else ""))
ax.set_title("Variance ratio VR(q) = Var(q-period return) / (q × Var(1-period return))")
ax.set_xlabel("horizon q (periods)")
ax.set_ylabel("VR(q)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
months = np.arange(1, 49)
axes[0].plot(months, r_true[:48], "o-", markersize=3, label="true")
axes[0].plot(months, r_obs[:48], "o-", markersize=3, label="observed (smoothed)")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0].set_title("Real estate: monthly returns, first 4 years")
axes[0].set_xlabel("month")
axes[0].legend()
axes[1].hist(r2_lev, bins=40, histtype="step", linewidth=1.5, label="levels")
axes[1].hist(r2_ret, bins=40, histtype="step", linewidth=1.5, label="returns")
axes[1].set_title("R² of regressions between independent random walks")
axes[1].set_xlabel("R²")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Under a random walk with daily volatility 1.2%, what are the 21-day and 10-day volatilities?
2. With 120 monthly returns, the sample lag-1 autocorrelation is 0.15. Is it significant on its own at 5%?
3. A hedge fund index has monthly lag-1 autocorrelation 0.35, reported annual volatility 4% and Sharpe ratio 1.5. **Interpret** these numbers
   and estimate the unsmoothed volatility and Sharpe ratio, assuming the smoothing model of section 2.

### Intermediate

4. Derive $\gamma_0 = \sigma^2/(1-\phi^2)$ and $\rho_k = \phi^k$ for the AR(1). Check them on a long simulated series with $\phi = 0.5$.
5. The Ljung–Box test assumes i.i.d. returns under the null. Compute its rejection rate (5% level, $m = 10$) for the GARCH series simulated 500
   times with 2,520 days each. Why might it reject more often than 5% although returns are uncorrelated?

### Advanced

6. Check the variance-ratio formula for $\phi = 0.2$ and $q = 12$ by simulating 2,000 series of 600 monthly returns and computing the ratio of the
   variance of 12-month sums to 12 times the variance of single months.
7. For the smoothed real estate series, compute the correlation with a simulated equity index (true correlation 0.5 between the true
   returns) before and after unsmoothing.

## 8. Solutions / Checks

In [ ]:
# 1.
print(f"1. 21-day vol {0.012 * np.sqrt(21):.2%}, 10-day vol {0.012 * np.sqrt(10):.2%}")

# 2.
print(f"2. band +/-{1.96 / np.sqrt(120):.3f}: 0.15 is inside")

# 3.
th = 0.35
vol_true = 0.04 * np.sqrt((1 + th) / (1 - th))
print(f"3. unsmoothed vol {vol_true:.2%}, Sharpe {1.5 * 0.04 / vol_true:.2f}")

# 4.
x4 = ar1_series(0.5, 200_000, rng)
print(f"4. var {x4.var():.4f} (theory {1 / (1 - 0.25):.4f}), acf {acf(x4, 4).round(3)} (theory {[0.5**k for k in range(1, 5)]})")

In [ ]:
# 5.
def garch_series(n_, rng_):
    h_ = sig_d**2
    out = np.empty(n_)
    for t in range(n_):
        out[t] = np.sqrt(h_) * rng_.standard_normal()
        h_ = omega + alpha_g * out[t] ** 2 + beta_g * h_
    return out

rej5 = np.mean([ljung_box(garch_series(2_520, rng), 10)[1] < 0.05 for _ in range(500)])
print(f"5. Ljung-Box rejection rate on GARCH returns: {rej5:.1%}")

# 6.
sums_ratio = []
for _ in range(2_000):
    x6 = ar1_series(0.2, 600, rng)
    s12 = x6.reshape(-1, 12).sum(axis=1)
    sums_ratio.append(s12.var(ddof=1) / (12 * x6.var(ddof=1)))
print(f"6. simulated VR(12) {np.mean(sums_ratio):.3f}, theory {1 + 2 * sum((1 - k / 12) * 0.2**k for k in range(1, 12)):.3f}")

# 7.
z1, z2 = rng.standard_normal(n_m), rng.standard_normal(n_m)
r_eq = 0.06 / 12 + 0.16 / np.sqrt(12) * z1
r_re_true = 0.05 / 12 + 0.12 / np.sqrt(12) * (0.5 * z1 + np.sqrt(0.75) * z2)
r_re_obs = np.empty(n_m)
r_re_obs[0] = r_re_true[0]
for t in range(1, n_m):
    r_re_obs[t] = (1 - theta) * r_re_true[t] + theta * r_re_obs[t - 1]
th7 = acf(r_re_obs, 1)[0]
r_re_uns = (r_re_obs[1:] - th7 * r_re_obs[:-1]) / (1 - th7)
print(f"7. corr with equities: true {np.corrcoef(r_eq, r_re_true)[0, 1]:.2f}, observed {np.corrcoef(r_eq, r_re_obs)[0, 1]:.2f}, "
      f"unsmoothed {np.corrcoef(r_eq[1:], r_re_uns)[0, 1]:.2f}")

**1.** $1.2\%\sqrt{21} = 5.50\%$ and $1.2\%\sqrt{10} = 3.79\%$.

**2.** The band is $\pm 1.96/\sqrt{120} = \pm 0.179$; 0.15 is inside, not significant on its own.

**3.** An autocorrelation of 0.35 in a hedge fund index points to stale or smoothed prices, not to predictable returns. Under
the smoothing model the true volatility is $4\%\sqrt{1.35/0.65} = 5.76\%$ and the Sharpe ratio $1.5\cdot 4/5.76 = 1.04$. The reported numbers overstate the
risk-adjusted performance and understate risk, as well as the correlation with listed markets.

**4.** From $x_t = \phi x_{t-1} + \varepsilon_t$: $\gamma_0 = \phi^2\gamma_0 + \sigma^2$, so $\gamma_0 = \sigma^2/(1-\phi^2)$; and $\gamma_k = \phi\gamma_{k-1}$, so $\rho_k = \phi^k$. The simulation agrees.

**5.** The rejection rate is 45%, far above 5%. Under GARCH the returns are uncorrelated but not independent: the variance of $\hat\rho_k$ is larger
than $1/n$ when volatility clusters, so the $\chi^2$ reference distribution is too narrow. Robust versions scale each $\hat\rho_k$ by a
heteroskedasticity-consistent variance.

**6.** The simulated ratio is 1.441 against the theoretical 1.448.

**7.** The correlation of the true returns with equities is 0.54; smoothing lowers the measured value to 0.42; unsmoothing restores 0.54. (A complete treatment would also
include the lagged correlation that smoothing creates.)

## 9. Conclusion

- Under a random walk, returns are i.i.d. and variance grows linearly with the horizon; returns are stationary, prices are not.
- Sample autocorrelations of an i.i.d. series have standard error $1/\sqrt n$; Ljung–Box tests many lags jointly with a $\chi^2_m$ reference.
- An AR(1) has $\rho_k = \phi^k$; the variance ratio $1 + 2\sum(1 - k/q)\rho_k$ measures how far $\sqrt T$ scaling is off.
- Smoothed returns understate volatility by $\sqrt{(1-\theta)/(1+\theta)}$ and overstate Sharpe ratios; unsmoothing inverts the filter.
- Regressions between price levels are spurious; use returns.
- Returns can be uncorrelated while squared returns are strongly autocorrelated: volatility clusters.

Next: [09/02 Volatility clustering, EWMA volatility](./02_volatility_clustering_ewma_volatility.ipynb).